In [1]:
# intalling pakages required
import subprocess
import sys

packages = [
    "pandas==2.2.0",
    "numpy==1.26.4",
    "scikit-learn==1.4.0",
    "matplotlib==3.8.2",
    "seaborn==0.13.2",
    "imbalanced-learn==0.12.0",
    "joblib==1.3.2"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed.")

All packages installed.


In [3]:
#Loading all libraries and sets up plotting configuration
%matplotlib inline
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
import joblib

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

plt.rcParams.update({
    "figure.figsize"    : (14, 6),
    "font.size"         : 12,
    "axes.titlesize"    : 14,
    "axes.titleweight"  : "bold",
    "axes.labelsize"    : 12,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "figure.dpi"        : 120,
    "savefig.dpi"       : 150,
    "savefig.bbox"      : "tight"
})

sns.set_style("whitegrid")

COLORS = {
    "no_default" : "#2ecc71",
    "default"    : "#e74c3c",
    "primary"    : "#3498db",
    "secondary"  : "#9b59b6",
    "warning"    : "#f39c12",
    "dark"       : "#2c3e50"
}

os.makedirs("../reports/figures", exist_ok=True)
os.makedirs("../models",          exist_ok=True)

print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"sklearn      : {__import__('sklearn').__version__}")

pandas       : 2.2.0
numpy        : 1.26.4
sklearn      : 1.4.0


In [ ]:
# Loading the cleaned CSV file 
df = pd.read_csv("../data/credit_default_cleaned.csv")

print(f"Shape   : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns : {list(df.columns)}")

Shape   : 29,965 rows x 24 columns
Columns : ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default']


In [7]:
#checking before touching any data
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Duplicates     : {df.duplicated().sum()}")
print(f"Target split   :")
print(df["default"].value_counts())

Missing values : 0
Duplicates     : 0
Target split   :
default
0    23335
1     6630
Name: count, dtype: int64


In [9]:
# Fix Invalid Category Codes Remaps invalid codes in 1_eda.
# EDUCATION codes 0, 5, 6 become 4 (Others).
# MARRIAGE code 0 becomes 3 (Others).
print("BEFORE:")
print(f"  EDUCATION : {sorted(df['EDUCATION'].unique())}")
print(f"  MARRIAGE  : {sorted(df['MARRIAGE'].unique())}")

df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"]  = df["MARRIAGE"].replace({0: 3})

print("\nAFTER:")
print(f"  EDUCATION : {sorted(df['EDUCATION'].unique())}")
print(f"  MARRIAGE  : {sorted(df['MARRIAGE'].unique())}")

print("\nEDUCATION counts:")
edu_map = {1: "Graduate", 2: "University", 3: "High School", 4: "Others"}
for val, count in df["EDUCATION"].value_counts().sort_index().items():
    print(f"  {val} {edu_map[val]:<12} : {count:,}")

print("\nMARRIAGE counts:")
mar_map = {1: "Married", 2: "Single", 3: "Others"}
for val, count in df["MARRIAGE"].value_counts().sort_index().items():
    print(f"  {val} {mar_map[val]:<10} : {count:,}")

BEFORE:
  EDUCATION : [1, 2, 3, 4]
  MARRIAGE  : [1, 2, 3]

AFTER:
  EDUCATION : [1, 2, 3, 4]
  MARRIAGE  : [1, 2, 3]

EDUCATION counts:
  1 Graduate     : 10,563
  2 University   : 14,019
  3 High School  : 4,915
  4 Others       : 468

MARRIAGE counts:
  1 Married    : 13,643
  2 Single     : 15,945
  3 Others     : 377


In [11]:
# Encode Categorical Variables
# Converts SEX, EDUCATION, MARRIAGE from category numbers into model-ready encoded values.
le = LabelEncoder()

cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]

print(f"{'Column':<15} {'Before values':<30} {'After values'}")
print("-" * 65)

for col in cat_cols:
    before = sorted(df[col].unique())
    df[col] = le.fit_transform(df[col])
    after   = sorted(df[col].unique())
    print(f"{col:<15} {str(before):<30} {str(after)}")

print(f"\nShape after encoding : {df.shape}")

Column          Before values                  After values
-----------------------------------------------------------------
SEX             [0, 1]                         [0, 1]
EDUCATION       [0, 1, 2, 3]                   [0, 1, 2, 3]
MARRIAGE        [0, 1, 2]                      [0, 1, 2]

Shape after encoding : (29965, 24)


In [13]:
# Handle Outliers Caps extreme values in BILL_AMT and PAY_AMT columns at the 99th percentile. 
# This stops extreme values from distorting the model.
bill_cols = ["BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6"]
pay_cols  = ["PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"]

cols_to_cap = bill_cols + pay_cols

print(f"{'Column':<15} {'Before Max':>15} {'99th Pct':>15} {'After Max':>15}")

for col in cols_to_cap:
    before_max = df[col].max()
    pct_99     = df[col].quantile(0.99)
    df[col]    = df[col].clip(upper=pct_99)
    after_max  = df[col].max()
    print(f"{col:<15} {before_max:>15,.0f} {pct_99:>15,.0f} {after_max:>15,.0f}")

print(f"\n Outliers capped in {len(cols_to_cap)} columns")

Column               Before Max        99th Pct       After Max
BILL_AMT1               350,134         350,119         350,119
BILL_AMT2               337,505         337,499         337,499
BILL_AMT3               325,254         325,107         325,107
BILL_AMT4               305,007         305,000         305,000
BILL_AMT5               285,880         285,872         285,872
BILL_AMT6               279,577         279,530         279,530
PAY_AMT1                 66,843          66,632          66,632
PAY_AMT2                 76,652          76,651          76,651
PAY_AMT3                 70,000          70,000          70,000
PAY_AMT4                 67,140          67,084          67,084
PAY_AMT5                 65,627          65,614          65,614
PAY_AMT6                 82,761          82,667          82,667

 Outliers capped in 12 columns


In [15]:
# Create Derived Features adds 6 new columns from existing data using credit risk domain knowledge.
# These new features capture patterns that raw columns cannot express alone.
print(f"Shape before : {df.shape}")

total_bill  = df[["BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6"]].sum(axis=1)
total_pay   = df[["PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"]].sum(axis=1)

df["credit_utilization_rate"] = df["BILL_AMT1"] / (df["LIMIT_BAL"] + 1)
df["debt_to_income_ratio"]    = total_bill      / (total_pay + 1)
df["avg_payment_delay"]       = df[["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]].mean(axis=1)
df["total_bill_amt"]          = total_bill
df["total_pay_amt"]           = total_pay
df["payment_to_bill_ratio"]   = total_pay       / (total_bill + 1)

print(f"Shape after  : {df.shape}")
print(f"\nNew features added:")
new_cols = ["credit_utilization_rate","debt_to_income_ratio",
            "avg_payment_delay","total_bill_amt",
            "total_pay_amt","payment_to_bill_ratio"]
print(df[new_cols].describe().round(3))

Shape before : (29965, 30)
Shape after  : (29965, 30)

New features added:
       credit_utilization_rate  debt_to_income_ratio  avg_payment_delay  \
count                29965.000             29965.000          29965.000   
mean                     0.422               596.744             -0.181   
std                      0.409             16378.279              0.981   
min                     -0.620            -38807.000             -2.000   
25%                      0.022                 1.436             -0.833   
50%                      0.315                 9.615              0.000   
75%                      0.823                23.070              0.000   
max                      6.455           1883126.730              6.000   

       total_bill_amt  total_pay_amt  payment_to_bill_ratio  
count       29965.000      29965.000              29965.000  
mean       265371.172      27707.875                 16.992  
std        355412.294      36634.046                836.952  
m

In [17]:
# Visualize New Features Plots distribution of the 4 most important new features split by default vs no default.
# Confirms they are useful predictors.
new_features = [
("credit_utilization_rate", "Credit Utilization Rate"),
("debt_to_income_ratio",    "Debt to Income Ratio"),
("avg_payment_delay",       "Average Payment Delay"),
("payment_to_bill_ratio",   "Payment to Bill Ratio")
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor("white")
axes = axes.flatten()

for idx, (col, title) in enumerate(new_features):
        ax     = axes[idx]
        upper  = df[col].quantile(0.95)
        no_def = df[(df["default"] == 0) & (df[col] <= upper)][col]
        def_   = df[(df["default"] == 1) & (df[col] <= upper)][col]
        ax.hist(no_def, bins=40, alpha=0.65,
        color=COLORS["no_default"], label="No Default",
        density=True, edgecolor="white", linewidth=0.3)
        ax.hist(def_,   bins=40, alpha=0.65,
        color=COLORS["default"],    label="Default",
        density=True, edgecolor="white", linewidth=0.3)
        ax.axvline(no_def.mean(), color=COLORS["no_default"],
        linestyle="--", linewidth=2,
        label=f"Mean: {no_def.mean():.2f}")
        ax.axvline(def_.mean(),   color=COLORS["default"],
        linestyle="--", linewidth=2,
        label=f"Mean: {def_.mean():.2f}")

        ax.set_title(title, fontsize=12, pad=10)
        ax.set_xlabel(col,  fontsize=10)
        ax.set_ylabel("Density", fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.set_facecolor("white")

fig.suptitle("New Derived Features: Default vs No Default",
        fontsize=15, fontweight="bold")
plt.tight_layout()
fig.savefig("../reports/figures/09_derived_features.png",
        dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)

print("Saved : reports/figures/09_derived_features.png ")

Saved : reports/figures/09_derived_features.png 


In [19]:
# Scale Features
# Applies StandardScaler to all numerical columns so large-value columns don't dominate the model over small-value columns.
cols_to_scale = [
    "LIMIT_BAL", "AGE",
    "BILL_AMT1","BILL_AMT2","BILL_AMT3",
    "BILL_AMT4","BILL_AMT5","BILL_AMT6",
    "PAY_AMT1","PAY_AMT2","PAY_AMT3",
    "PAY_AMT4","PAY_AMT5","PAY_AMT6",
    "credit_utilization_rate",
    "debt_to_income_ratio",
    "avg_payment_delay",
    "total_bill_amt",
    "total_pay_amt",
    "payment_to_bill_ratio"
]

scaler    = StandardScaler()
df_scaled = df.copy()

df_scaled[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print(f"{'Column':<30} {'Before Mean':>12} {'After Mean':>12} {'After Std':>12}")
print("-" * 70)
for col in cols_to_scale[:8]:
    print(f"{col:<30} {df[col].mean():>12.2f} "
        f"{df_scaled[col].mean():>12.5f} "
        f"{df_scaled[col].std():>12.5f}")
print(f"  ... ({len(cols_to_scale)} total columns scaled)")
print(f"\n All {len(cols_to_scale)} columns scaled to mean≈0 std≈1")

Column                          Before Mean   After Mean    After Std
----------------------------------------------------------------------
LIMIT_BAL                         167442.01     -0.00000      1.00002
AGE                                   35.49      0.00000      1.00002
BILL_AMT1                          50460.37     -0.00000      1.00002
BILL_AMT2                          48411.87      0.00000      1.00002
BILL_AMT3                          46199.21     -0.00000      1.00002
BILL_AMT4                          42525.85     -0.00000      1.00002
BILL_AMT5                          39594.61      0.00000      1.00002
BILL_AMT6                          38179.26     -0.00000      1.00002
  ... (20 total columns scaled)

 All 20 columns scaled to mean≈0 std≈1


In [21]:
# Splits data into Train (70%), Validation (15%), and Test (15%) sets using stratified splitting so each set has the same default rate.
X = df_scaled.drop("default", axis=1)
y = df_scaled["default"]

print(f"Features (X) : {X.shape}")
print(f"Target   (y) : {y.shape}")

X_temp,  X_test,  y_temp,  y_test  = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val,   y_train, y_val   = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print(f"\n{'Split':<12} {'Rows':>8} {'Pct':>8} {'Defaults':>10} {'Default%':>10}")
print("-" * 52)
print(f"{'Train':<12} {len(X_train):>8,} "
    f"{len(X_train)/len(X)*100:>7.1f}% "
    f"{int(y_train.sum()):>10,} "
    f"{y_train.mean()*100:>9.1f}%")
print(f"{'Validation':<12} {len(X_val):>8,} "
    f"{len(X_val)/len(X)*100:>7.1f}% "
    f"{int(y_val.sum()):>10,} "
    f"{y_val.mean()*100:>9.1f}%")
print(f"{'Test':<12} {len(X_test):>8,} "
    f"{len(X_test)/len(X)*100:>7.1f}% "
    f"{int(y_test.sum()):>10,} "
    f"{y_test.mean()*100:>9.1f}%")
print(f"{'Total':<12} {len(X):>8,} {'100%':>8} "
    f"{int(y.sum()):>10,} "
    f"{y.mean()*100:>9.1f}%")

Features (X) : (29965, 29)
Target   (y) : (29965,)

Split            Rows      Pct   Defaults   Default%
----------------------------------------------------
Train          20,987    70.0%      4,643      22.1%
Validation      4,483    15.0%        992      22.1%
Test            4,495    15.0%        995      22.1%
Total          29,965     100%      6,630      22.1%


In [23]:
# Confirms each split has same class distribution and shows row counts visually.
splits      = ["Train",    "Validation", "Test"]
split_sizes = [len(X_train), len(X_val),  len(X_test)]
def_rates   = [y_train.mean()*100, y_val.mean()*100, y_test.mean()*100]
split_colors= [COLORS["primary"], COLORS["secondary"], COLORS["warning"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor("white")

bars1 = axes[0].bar(splits, split_sizes,
                    color=split_colors,
                    edgecolor="white", linewidth=1.5, width=0.5)
for bar, size in zip(bars1, split_sizes):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 80,
        f"{size:,}",
        ha="center", va="bottom",
        fontweight="bold", fontsize=12
    )
axes[0].set_title("Rows per Split")
axes[0].set_ylabel("Number of Rows")
axes[0].set_ylim(0, max(split_sizes) * 1.2)
axes[0].set_facecolor("white")

bars2 = axes[1].bar(splits, def_rates,
                    color=split_colors,
                    edgecolor="white", linewidth=1.5, width=0.5)
for bar, rate in zip(bars2, def_rates):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f"{rate:.1f}%",
        ha="center", va="bottom",
        fontweight="bold", fontsize=12
    )
axes[1].set_title("Default Rate per Split")
axes[1].set_ylabel("Default Rate (%)")
axes[1].set_ylim(0, 35)
axes[1].axhline(y=y.mean()*100, color="navy",
                linestyle="--", linewidth=2,
                label=f"Overall: {y.mean()*100:.1f}%")
axes[1].legend(fontsize=10)
axes[1].set_facecolor("white")

fig.suptitle("Train / Validation / Test Split",
            fontsize=15, fontweight="bold")
plt.tight_layout()
fig.savefig("../reports/figures/10_data_split.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)

print("Saved : reports/figures/10_data_split.png ")

Saved : reports/figures/10_data_split.png 


In [25]:
# Saves everything 3_ needs — 6 split CSV files, the scaler, and the full processed dataframe.
X_train.to_csv("../data/X_train.csv", index=False)
X_val.to_csv(  "../data/X_val.csv",   index=False)
X_test.to_csv( "../data/X_test.csv",  index=False)

y_train.to_csv("../data/y_train.csv", index=False)
y_val.to_csv(  "../data/y_val.csv",   index=False)
y_test.to_csv( "../data/y_test.csv",  index=False)

df_scaled.to_csv("../data/credit_default_processed.csv", index=False)

joblib.dump(scaler, "../models/scaler.pkl")

saved_files = [
    ("../data/X_train.csv",                  "Train features"),
    ("../data/X_val.csv",                    "Validation features"),
    ("../data/X_test.csv",                   "Test features"),
    ("../data/y_train.csv",                  "Train labels"),
    ("../data/y_val.csv",                    "Validation labels"),
    ("../data/y_test.csv",                   "Test labels"),
    ("../data/credit_default_processed.csv", "Full processed data"),
    ("../models/scaler.pkl",                 "Fitted scaler")
]

print(f"{'File':<40} {'Size':>12}")
print("-" * 55)
for path, label in saved_files:
    size = os.path.getsize(path)
    print(f"{label:<40} {size:>10,} bytes")

File                                             Size
-------------------------------------------------------
Train features                            8,855,463 bytes
Validation features                       1,891,732 bytes
Test features                             1,896,941 bytes
Train labels                                 62,970 bytes
Validation labels                            13,458 bytes
Test labels                                  13,494 bytes
Full processed data                      12,703,470 bytes
Fitted scaler                                 1,751 bytes


In [32]:
import os
import pandas as pd

print("=" * 55)
print("PHASE 2 COMPLETION CHECK")
print("=" * 55)

required_files = {
    "../data/X_train.csv"                        : "Train features",
    "../data/X_val.csv"                          : "Validation features",
    "../data/X_test.csv"                         : "Test features",
    "../data/y_train.csv"                        : "Train labels",
    "../data/y_val.csv"                          : "Validation labels",
    "../data/y_test.csv"                         : "Test labels",
    "../data/credit_default_processed.csv"       : "Full processed data",
    "../models/scaler.pkl"                       : "Fitted scaler",
    "../reports/figures/09_derived_features.png" : "Derived features chart",
    "../reports/figures/10_data_split.png"       : "Data split chart"
}

all_good = True
for path, label in required_files.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"  ✅  {label:<35} {size:>12,} bytes")
    else:
        print(f"  ❌  {label:<35} NOT FOUND")
        all_good = False

X_train_c = pd.read_csv("../data/X_train.csv")
X_val_c   = pd.read_csv("../data/X_val.csv")
X_test_c  = pd.read_csv("../data/X_test.csv")
y_train_c = pd.read_csv("../data/y_train.csv").squeeze()
y_val_c   = pd.read_csv("../data/y_val.csv").squeeze()
y_test_c  = pd.read_csv("../data/y_test.csv").squeeze()

print(f"\n{'Split':<12} {'Rows':>8} {'Features':>10} {'Default%':>10}")
print("-" * 44)
print(f"{'Train':<12} {len(X_train_c):>8,} "
    f"{X_train_c.shape[1]:>10} "
    f"{y_train_c.mean()*100:>9.1f}%")
print(f"{'Validation':<12} {len(X_val_c):>8,} "
    f"{X_val_c.shape[1]:>10} "
    f"{y_val_c.mean()*100:>9.1f}%")
print(f"{'Test':<12} {len(X_test_c):>8,} "
    f"{X_test_c.shape[1]:>10} "
    f"{y_test_c.mean()*100:>9.1f}%")

print("\n" + "=" * 55)
print(f"  PHASE 2 : {'✅ 100% COMPLETE' if all_good else '⚠️ FIX ITEMS ABOVE'}")
print("=" * 55)

PHASE 2 COMPLETION CHECK
  ✅  Train features                         8,855,463 bytes
  ✅  Validation features                    1,891,732 bytes
  ✅  Test features                          1,896,941 bytes
  ✅  Train labels                              62,970 bytes
  ✅  Validation labels                         13,458 bytes
  ✅  Test labels                               13,494 bytes
  ✅  Full processed data                   12,703,470 bytes
  ✅  Fitted scaler                              1,751 bytes
  ✅  Derived features chart                   175,792 bytes
  ✅  Data split chart                          68,064 bytes

Split            Rows   Features   Default%
--------------------------------------------
Train          20,987         29      22.1%
Validation      4,483         29      22.1%
Test            4,495         29      22.1%

  PHASE 2 : ✅ 100% COMPLETE
